In [9]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, roc_auc_score,
)
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "Malgun Gothic" 
matplotlib.rcParams["axes.unicode_minus"] = False

# 데이터 로드
master_df = pd.read_csv("data/master_dataset.csv", index_col=0, parse_dates=True)
STOCK_COLS = ["삼성전자", "SK하이닉스", "NAVER", "KODEX200", "KODEX채권"]
WEIGHTS = np.array([0.25, 0.20, 0.20, 0.25, 0.10])
CONFIDENCE = 0.95

portfolio_returns = (master_df[STOCK_COLS] * WEIGHTS).sum(axis=1)

In [ ]:
# 1. 피처 엔지니어링
# ════════════════════════════════════════════════════════════════
def build_features(master_df: pd.DataFrame, port_returns: pd.Series) -> pd.DataFrame:
    df = master_df.copy()
    df["portfolio_return"] = port_returns

    # 수익률 피처
    for col in STOCK_COLS:
        # 롤링 변동성 (5일, 20일)
        df[f"{col}_vol5"]  = df[col].rolling(5).std()
        df[f"{col}_vol20"] = df[col].rolling(20).std()
        # 모멘텀 (5일 누적 수익률)
        df[f"{col}_mom5"]  = df[col].rolling(5).sum()

    # 포트폴리오 수준 피처 
    df["port_vol5"]      = port_returns.rolling(5).std()
    df["port_vol20"]     = port_returns.rolling(20).std()
    df["port_mom5"]      = port_returns.rolling(5).sum()
    df["port_skew20"]    = port_returns.rolling(20).skew()   
    df["port_kurt20"]    = port_returns.rolling(20).kurt()

    # 거시경제 피처
    df["rate_spread"]    = df["국고채3년"] - df["기준금리"]    # 금리 스프레드
    df["vix_change"]     = df["VIX"].pct_change()              # VIX 변화율
    df["vix_ma5"]        = df["VIX"].rolling(5).mean()         # VIX 5일 평균

    return df.dropna()

feat_df = build_features(master_df, portfolio_returns)
print(f"피처 생성 완료: {feat_df.shape[0]}일 x {feat_df.shape[1]}컬럼")


피처 생성 완료: 963일 x 32컬럼


In [11]:
# 2. 타겟 변수 생성
# "다음날 포트폴리오 수익률이 히스토리컬 VaR 임계값을 초과하는 손실인가?"
# 1 = VaR 초과 손실 (위험) / 0 = 정상
# ════════════════════════════════════════════════════════════════
var_threshold = np.percentile(portfolio_returns, (1 - CONFIDENCE) * 100)
print(f"\nVaR 임계값 (95%): {var_threshold:.4f} ({var_threshold*100:.2f}%)")

# 다음날 수익률이 임계값 이하 = 위험(1), 아니면 정상(0)
feat_df["target"] = (feat_df["portfolio_return"].shift(-1) <= var_threshold).astype(int)
feat_df = feat_df.dropna()

pos_ratio = feat_df["target"].mean()
print(f"위험 발생 비율: {pos_ratio:.2%} ({feat_df['target'].sum()}일 / {len(feat_df)}일)")


VaR 임계값 (95%): -0.0198 (-1.98%)
위험 발생 비율: 4.98% (48일 / 963일)


In [14]:
# 3. 학습/테스트 분할
# ════════════════════════════════════════════════════════════════
FEATURE_COLS = [c for c in feat_df.columns if c not in ["target", "portfolio_return"]]

X = feat_df[FEATURE_COLS]
y = feat_df["target"]

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\n학습 기간: {X_train.index[0].date()} ~ {X_train.index[-1].date()} ({len(X_train)}일)")
print(f"테스트 기간: {X_test.index[0].date()} ~ {X_test.index[-1].date()} ({len(X_test)}일)")



학습 기간: 2021-02-01 ~ 2024-03-15 (770일)
테스트 기간: 2024-03-18 ~ 2024-12-30 (193일)


In [15]:
# 4. LightGBM 
# ════════════════════════════════════════════════════════════════
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=15,
    min_child_samples=20,
    class_weight="balanced", 
    random_state=42,
    verbose=-1
)

tscv = TimeSeriesSplit(n_splits=5)
cv_aucs = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model.fit(Xtr, ytr)
    prob = model.predict_proba(Xval)[:, 1]
    auc  = roc_auc_score(yval, prob)
    cv_aucs.append(auc)
    print(f"  Fold {fold+1} AUC: {auc:.4f}")

print(f"\n  CV 평균 AUC: {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")

model.fit(X_train, y_train)



  Fold 1 AUC: 0.3488
  Fold 2 AUC: 0.4313
  Fold 3 AUC: 0.4398
  Fold 4 AUC: 0.6439
  Fold 5 AUC: 0.5888

  CV 평균 AUC: 0.4905 ± 0.1090


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,4
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
# 5. 테스트 성능 평가
# ════════════════════════════════════════════════════════════════
y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]
test_auc    = roc_auc_score(y_test, y_prob)

print(f"\n테스트 세트 성능")
print(f"  AUC-ROC: {test_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=["정상", "VaR초과"]))


print(f"\nAUC: {test_auc:.4f}")

# F1 기준
from sklearn.metrics import f1_score, recall_score, precision_score

best_thresh, best_f1 = 0.5, 0.0
print("\n임계값 탐색")
for thresh in np.arange(0.05, 0.50, 0.05):
    pred_t = (y_prob >= thresh).astype(int)
    r  = recall_score(y_test, pred_t, zero_division=0)
    p  = precision_score(y_test, pred_t, zero_division=0)
    f1 = f1_score(y_test, pred_t, zero_division=0)
    print(f"  임계값 {thresh:.2f} → Recall: {r:.2%}, Precision: {p:.2%}, F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1    = f1
        best_thresh = thresh

print(f"\n최적 임계값: {best_thresh:.2f}  (F1: {best_f1:.4f})")

# ── 최적 임계값으로 최종 예측 ─────────────────────────────────
y_pred_best = (y_prob >= best_thresh).astype(int)


테스트 세트 성능
  AUC-ROC: 0.7362
              precision    recall  f1-score   support

          정상       0.94      1.00      0.97       181
       VaR초과       0.00      0.00      0.00        12

    accuracy                           0.94       193
   macro avg       0.47      0.50      0.48       193
weighted avg       0.88      0.94      0.91       193


AUC: 0.7362

임계값 탐색
  임계값 0.05 → Recall: 25.00%, Precision: 33.33%, F1: 0.2857
  임계값 0.10 → Recall: 16.67%, Precision: 50.00%, F1: 0.2500
  임계값 0.15 → Recall: 8.33%, Precision: 33.33%, F1: 0.1333
  임계값 0.20 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000
  임계값 0.25 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000
  임계값 0.30 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000
  임계값 0.35 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000
  임계값 0.40 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000
  임계값 0.45 → Recall: 0.00%, Precision: 0.00%, F1: 0.0000

최적 임계값: 0.05  (F1: 0.2857)


c:\Users\ilhoo\anaconda3\envs\finrisk\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ilhoo\anaconda3\envs\finrisk\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ilhoo\anaconda3\envs\finrisk\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [17]:
# 6. 전통 VaR vs ML VaR 비교
# ════════════════════════════════════════════════════════════════
print(f"\n 전통 VaR vs ML VaR 비교")
trad_pred  = np.zeros(len(y_test), dtype=int)
ml_recall  = recall_score(y_test, y_pred_best, zero_division=0)
ml_prec    = precision_score(y_test, y_pred_best, zero_division=0)

print(f"  전통 VaR  — Recall:  0.00%  (임계값 기반, 사전 감지 불가)")
print(f"  ML VaR    — Recall: {ml_recall:.2%}  Precision: {ml_prec:.2%}")
print(f"  AUC-ROC:  {test_auc:.4f}  (0.5 = 랜덤, 1.0 = 완벽)")
print(f"  → ML이 VaR 초과 리스크를 {ml_recall:.2%} 비율로 사전 포착")
print(f"\n최종 분류 리포트 (임계값 {best_thresh:.2f})")
print(classification_report(y_test, y_pred_best,
      target_names=["정상", "VaR초과"], zero_division=0))



 전통 VaR vs ML VaR 비교
  전통 VaR  — Recall:  0.00%  (임계값 기반, 사전 감지 불가)
  ML VaR    — Recall: 25.00%  Precision: 33.33%
  AUC-ROC:  0.7362  (0.5 = 랜덤, 1.0 = 완벽)
  → ML이 VaR 초과 리스크를 25.00% 비율로 사전 포착

최종 분류 리포트 (임계값 0.05)
              precision    recall  f1-score   support

          정상       0.95      0.97      0.96       181
       VaR초과       0.33      0.25      0.29        12

    accuracy                           0.92       193
   macro avg       0.64      0.61      0.62       193
weighted avg       0.91      0.92      0.92       193



In [ ]:
# 7. SHAP — 피처 중요도 설명 (XAI)
# ════════════════════════════════════════════════════════════════
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values[1] if isinstance(shap_values, list) else shap_values,
    X_test,
    plot_type="bar",
    show=False,
    max_display=15
)
plt.title("SHAP Feature Importance — VaR 초과 리스크 예측")
plt.tight_layout()
plt.savefig("data/shap_importance.png", dpi=150)
plt.close()

result_df = X_test.copy()
result_df["actual"]        = y_test
result_df["ml_pred"]      = y_pred_best  
result_df["ml_prob_risk"] = y_prob
result_df["threshold"]    = best_thresh

result_df[["actual", "ml_pred", "ml_prob_risk"]].to_csv("data/ml_predictions.csv")

print(f"\n AUC: {test_auc:.4f}")

c:\Users\ilhoo\anaconda3\envs\finrisk\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



 AUC: 0.7362


In [19]:
import joblib
joblib.dump(model, "data/model.pkl")
print("모델 저장")

모델 저장
